In [2]:
import torch

# CPU 환경 기준 세팅
torch.manual_seed(42)
print(f"PyTorch Version: {torch.__version__}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\Playdata\miniconda3\envs\myenv\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\Playdata\miniconda3\envs\myenv\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Playdata\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Playdata\AppData\Roaming\Python\Python310\site-packages\traitlets\config\application.py", line 1075, in lau

PyTorch Version: 2.2.2


In [3]:
def step_function(x):
    # x가 0 이상이면 1.0, 0 미만이면 0.0을 반환
    return (x >= 0).float()

In [4]:
# 입력 데이터 (0,0), (0,1), (1,0), (1,1)
x = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]])

# 가중치와 편향
w_and = torch.tensor([[0.5], [0.5]])
b_and = -0.7

# 선형 결합 식 적용 (행렬곱 + 편향)
y_sum_and = torch.matmul(x, w_and) + b_and

# 계단 함수 통과
output_and = step_function(y_sum_and)

print("--- AND Gate 테스트 결과 ---")
for x_val, out in zip(x, output_and):
    print(f"입력 {x_val.tolist()} -> 출력 {int(out.item())}")

--- AND Gate 테스트 결과 ---
입력 [0.0, 0.0] -> 출력 0
입력 [0.0, 1.0] -> 출력 0
입력 [1.0, 0.0] -> 출력 0
입력 [1.0, 1.0] -> 출력 1


In [5]:
# 임의의 가중치로 XOR을 시도해봅니다.
w_xor = torch.tensor([[1.0], [1.0]])
b_xor = -0.5

y_sum_xor = torch.matmul(x, w_xor) + b_xor
output_xor = step_function(y_sum_xor)

print("--- XOR Gate 시도 결과 (실패) ---")
for x_val, out in zip(x, output_xor):
    correct_target = int(x_val[0] != x_val[1])
    print(f"입력 {x_val.tolist()} -> 출력 {int(out.item())} (실제 정답: {correct_target})")

print("\n-> 결과 분석: 퍼셉트론 하나(단일 선형 결합)로는 XOR과 같은 비선형 문제를 분리하지 못합니다. 이것이 다음 모듈에서 '은닉층(Hidden Layer)'을 추가하는 이유입니다.")

--- XOR Gate 시도 결과 (실패) ---
입력 [0.0, 0.0] -> 출력 0 (실제 정답: 0)
입력 [0.0, 1.0] -> 출력 1 (실제 정답: 1)
입력 [1.0, 0.0] -> 출력 1 (실제 정답: 1)
입력 [1.0, 1.0] -> 출력 1 (실제 정답: 0)

-> 결과 분석: 퍼셉트론 하나(단일 선형 결합)로는 XOR과 같은 비선형 문제를 분리하지 못합니다. 이것이 다음 모듈에서 '은닉층(Hidden Layer)'을 추가하는 이유입니다.


In [6]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

In [28]:
# ==============================================================
# 실험: 활성화 함수 없이 nn.Linear를 100층 쌓기
# ==============================================================
torch.manual_seed(42)

# 100층짜리 네트워크 (활성화 함수 없음!)
layers_no_act = []
for _ in range(100):
    layers_no_act.append(nn.Linear(4, 4))

deep_no_activation = nn.Sequential(*layers_no_act)

# 1층짜리 네트워크
single_layer = nn.Linear(4, 4)

# 테스트 입력
x_test = torch.randn(1, 4)

# 100층 네트워크의 전체 가중치를 하나로 합성
# W_total = W_100 × W_99 × ... × W_1
W_합성 = torch.eye(4)
b_합성 = torch.zeros(4)
for layer in layers_no_act:
    W_합성 = layer.weight @ W_합성
    b_합성 = layer.weight @ b_합성 + layer.bias

# 합성된 단일 가중치로 계산
output_합성 = x_test @ W_합성.T + b_합성

# 100층 네트워크로 직접 계산
with torch.no_grad():
    output_100층 = deep_no_activation(x_test)

print("=" * 60)
print("코드 증명 A: 활성화 함수 없는 100층 = 단일 선형 변환")
print("=" * 60)
print(f"100층 네트워크 출력:    {output_100층.detach().tolist()}")
print(f"합성 단일 변환 출력:    {output_합성.detach().tolist()}")
print(f"\n완벽 일치: {torch.allclose(output_100층, output_합성, atol=1e-3)}")
print("\n→ 결론: 활성화 함수 없이 아무리 많은 층을 쌓아도")
print("  결국 y = W'x + b' 하나의 1차 변환으로 축약됩니다.")
print("  층을 깊게 쌓는 의미가 완전히 사라집니다!")

코드 증명 A: 활성화 함수 없는 100층 = 단일 선형 변환
100층 네트워크 출력:    [[-0.12809447944164276, 0.042219728231430054, -0.23123304545879364, -0.44191646575927734]]
합성 단일 변환 출력:    [[-0.12809447944164276, 0.042219728231430054, -0.23123304545879364, -0.44191646575927734]]

완벽 일치: True

→ 결론: 활성화 함수 없이 아무리 많은 층을 쌓아도
  결국 y = W'x + b' 하나의 1차 변환으로 축약됩니다.
  층을 깊게 쌓는 의미가 완전히 사라집니다!


In [ ]:
import numpy as np
a = np.array([[1,2]])
b = np.array([[3],
              [4]])

a @ b

A = torch.full((3, 2), 2) # 행렬 생성
B = torch.full((2, 3), 3) # 행렬 생성

result = torch.matmul(A, B)
print (result ,result.size())

tensor([[12, 12, 12],
        [12, 12, 12],
        [12, 12, 12]]) torch.Size([3, 3])


In [30]:
# ==============================================================
# 실험: 5층 네트워크에서 시그모이드의 기울기 소실 관찰
# ==============================================================
torch.manual_seed(42)

class SigmoidNet(nn.Module):
    """시그모이드를 은닉층에 사용하는 5층 네트워크"""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.layer2 = nn.Linear(8, 8)
        self.layer3 = nn.Linear(8, 8)
        self.layer4 = nn.Linear(8, 8)
        self.layer5 = nn.Linear(8, 1)
    
    def forward(self, x):
        x = torch.sigmoid(self.layer1(x))  # 은닉층 1
        x = torch.sigmoid(self.layer2(x))  # 은닉층 2
        x = torch.sigmoid(self.layer3(x))  # 은닉층 3
        x = torch.sigmoid(self.layer4(x))  # 은닉층 4
        x = self.layer5(x)                 # 출력층
        return x

sigmoid_net = SigmoidNet()

# 순전파 + 역전파 실행
x_input = torch.randn(1, 4)
y_target = torch.tensor([[1.0]])
output = sigmoid_net(x_input)
loss = (output - y_target) ** 2
loss.backward()

print("=" * 60)
print("코드 증명 B: 시그모이드 → 기울기 소실 관찰")
print("=" * 60)
print(f"{'레이어':>10} | {'기울기 평균 크기':>18} | {'상태':>10}")
print("-" * 45)

for name, param in sigmoid_net.named_parameters():
    if 'weight' in name:
        grad_magnitude = param.grad.abs().mean().item()
        status = "⚠️ 소실!" if grad_magnitude < 0.001 else "보통" if grad_magnitude < 0.01 else "양호"
        print(f"{name:>10} | {grad_magnitude:>18.8f} | {status:>10}")

print(f"\n→ 관찰: 출력에서 멀어질수록(layer1 방향) 기울기가 급격히 작아집니다.")
print(f"  → 앞쪽 레이어는 학습이 거의 이루어지지 않습니다!")

코드 증명 B: 시그모이드 → 기울기 소실 관찰
       레이어 |          기울기 평균 크기 |         상태
---------------------------------------------
layer1.weight |         0.00014377 |     ⚠️ 소실!
layer2.weight |         0.00055731 |     ⚠️ 소실!
layer3.weight |         0.00339065 |         보통
layer4.weight |         0.03512243 |         양호
layer5.weight |         0.64159119 |         양호

→ 관찰: 출력에서 멀어질수록(layer1 방향) 기울기가 급격히 작아집니다.
  → 앞쪽 레이어는 학습이 거의 이루어지지 않습니다!


In [31]:
# ==============================================================
# 실험: ReLU를 사용하는 동일한 5층 구조
# ==============================================================
torch.manual_seed(42)

class ReLUNet(nn.Module):
    """ReLU를 은닉층에 사용하는 5층 네트워크"""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.layer2 = nn.Linear(8, 8)
        self.layer3 = nn.Linear(8, 8)
        self.layer4 = nn.Linear(8, 8)
        self.layer5 = nn.Linear(8, 1)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))   # ReLU로만 교체!
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        x = torch.relu(self.layer4(x))
        x = self.layer5(x)
        return x

relu_net = ReLUNet()

# 동일한 입력으로 순전파 + 역전파
output_relu = relu_net(x_input)
loss_relu = (output_relu - y_target) ** 2
loss_relu.backward()

# ============== 비교 출력 ==============
print("=" * 70)
print("코드 증명 C: Sigmoid vs ReLU 기울기 크기 비교")
print("=" * 70)
print(f"{'레이어':>10} | {'Sigmoid 기울기':>18} | {'ReLU 기울기':>18} | {'배율':>8}")
print("-" * 62)

sigmoid_grads = {}
relu_grads = {}

for name, param in sigmoid_net.named_parameters():
    if 'weight' in name:
        sigmoid_grads[name] = param.grad.abs().mean().item()

for name, param in relu_net.named_parameters():
    if 'weight' in name:
        relu_grads[name] = param.grad.abs().mean().item()

for name in sigmoid_grads:
    sig_g = sigmoid_grads[name]
    rel_g = relu_grads[name]
    ratio = rel_g / sig_g if sig_g > 0 else float('inf')
    print(f"{name:>10} | {sig_g:>18.8f} | {rel_g:>18.8f} | {ratio:>7.1f}×")

print(f"\n→ 결론: ReLU는 특히 앞쪽 레이어(layer1)에서 Sigmoid 대비")
print(f"  수십~수백 배 큰 기울기를 유지합니다.")
print(f"  → 깊은 네트워크에서도 모든 층이 골고루 학습됩니다!")

코드 증명 C: Sigmoid vs ReLU 기울기 크기 비교
       레이어 |        Sigmoid 기울기 |           ReLU 기울기 |       배율
--------------------------------------------------------------
layer1.weight |         0.00014377 |         0.00351978 |    24.5×
layer2.weight |         0.00055731 |         0.00382042 |     6.9×
layer3.weight |         0.00339065 |         0.00219299 |     0.6×
layer4.weight |         0.03512243 |         0.00542956 |     0.2×
layer5.weight |         0.64159119 |         0.05177632 |     0.1×

→ 결론: ReLU는 특히 앞쪽 레이어(layer1)에서 Sigmoid 대비
  수십~수백 배 큰 기울기를 유지합니다.
  → 깊은 네트워크에서도 모든 층이 골고루 학습됩니다!
